# Modelado — comparación de modelos y bloques de features

**Objetivo.** Medir con **validación cruzada temporal** qué información aporta cada bloque de features (mercado, plantilla, forma, estadísticas de partido) y con qué modelos, antes de entrenar los modelos finales. Entrada: `dataset_final.csv`, `feature_sets.json` y `backtest_odds.csv`, generados por `scripts/notebooks/feature.ipynb`.

**Convención.** Cada celda de código va precedida por una celda de texto (qué vamos a hacer y por qué) y seguida por otra (qué encontramos y qué decidimos).

| Paso | Qué hace | Estado |
|---|---|---|
| 0 | Carga y diseño de validación temporal | hecho |
| 1 | Referencias: prior y mercado crudo | hecho |
| 2 | Aporte de cada bloque (LR y HistGradientBoosting) | hecho |
| 3 | Habilidades individuales del once titular (clasificación), con y sin la temporada 2024/25 | hecho |
| 4 | Regresión de la diferencia de goles (Lineal, Random Forest, Boosting y ensemble) | hecho |
| 5 | ¿Las habilidades aportan información sobre el mercado? (apilado con `MKT_logit`) | hecho |
| 6 | Selección de hiperparámetros de SVM, XGBoost y MLP | hecho |
| 7 | Evaluación de SVM, XGBoost, MLP y ensemble frente al mercado y la LR | hecho |
| 8 | Calibración de los modelos finales | hecho |
| 9 | **Selección de características: Mutual Information vs. PCA** (pregunta de investigación 3) | hecho |
| 10 | **Preguntas de investigación e hipótesis**: estabilidad temporal, plantilla vs. forma, inicio de temporada, calibración vs. dinero | hecho |

**Pregunta de investigación que guía el notebook.** ¿Alguna información distinta de las cuotas mejora la predicción fuera de muestra?

## Paso 0 — Carga y diseño de la validación temporal

**Qué vamos a hacer.** Cargar el dataset de modelado y definir el esquema de validación:

- *Ventana expansiva por temporada:* para cada temporada de validación (2018/19 … 2025/26) se entrena con todas las temporadas anteriores. Ningún fold ve el futuro y el mínimo de entrenamiento son 4 temporadas.
- *Target:* `FTR` ∈ {H, D, A}.
- *Métrica principal:* log loss (sensible a la calibración); secundaria: accuracy.
- *Conjuntos de features:* F1 (sin NaN; para LR, SVM, MLP) y F2 (completo, con NaN; para árboles).

**Por qué.** Un único split puede favorecer a una temporada; ocho folds dan una media y una dispersión.

In [1]:
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, accuracy_score

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
DATA = '../data/'
df = pd.read_csv(DATA + 'dataset_final.csv')
odds = pd.read_csv(DATA + 'backtest_odds.csv')
fs = json.load(open(DATA + 'feature_sets.json'))
F1, F2, TARGET = fs['F1_sin_nan'], fs['F2_completo'], fs['target']

assert (df[['Date', 'HomeTeam', 'AwayTeam']].values == odds[['Date', 'HomeTeam', 'AwayTeam']].values).all()
y = df[TARGET].map({'H': 0, 'D': 1, 'A': 2}).to_numpy()
seas = df['Season'].to_numpy()
FOLDS = [s for s in sorted(df['Season'].unique()) if s >= 1819]
n_tr = {int(s): int((seas < s).sum()) for s in FOLDS}
n_va = {int(s): int((seas == s).sum()) for s in FOLDS}
print('filas:', len(df), '| F1:', len(F1), '| F2:', len(F2))
print('folds (validacion):', [int(s) for s in FOLDS])
print('filas de entrenamiento por fold:', n_tr)
print('filas de validacion por fold   :', n_va)
print('distribucion del target:', df[TARGET].value_counts(normalize=True).round(3).to_dict())

filas: 36119 | F1: 13 | F2: 33
folds (validacion): [1819, 1920, 2021, 2122, 2223, 2324, 2425, 2526]
filas de entrenamiento por fold: {1819: 11936, 1920: 14920, 2021: 17721, 2122: 20884, 2223: 24007, 2324: 27064, 2425: 30120, 2526: 33138}
filas de validacion por fold   : {1819: 2984, 1920: 2801, 2021: 3163, 2122: 3123, 2223: 3057, 2324: 3056, 2425: 3018, 2526: 2981}
distribucion del target: {'H': 0.444, 'A': 0.307, 'D': 0.25}


**Hallazgo.** 36,119 partidos con F1 = 13 y F2 = 33 features. Hay 8 folds de validación (2018/19–2025/26); el primero entrena con 4 temporadas y el último con 11, y cada fold valida con unos 3,000 partidos. El target está desbalanceado (H 44.4%, D 25.0%, A 30.7%).

**Decisión.** Usar este esquema de validación para todas las comparaciones, de modo que prior, mercado y modelos se evalúen exactamente sobre los mismos partidos.

## Paso 1 — Referencias: prior y mercado crudo

**Qué vamos a hacer.** Calcular, en los mismos folds, dos referencias sin modelo:

1. **Prior:** las frecuencias de H/D/A del conjunto de entrenamiento.
2. **Mercado crudo:** las probabilidades implícitas del consenso de casas (`MKT_pH/pD/pA`), tal cual.

**Por qué.** Un modelo solo aporta valor si mejora el prior (hay señal) y, sobre todo, el mercado (hay información que las casas no usan).

In [2]:
mk = odds[['MKT_pH', 'MKT_pD', 'MKT_pA']].to_numpy()
filas = []
for s in FOLDS:
    tr, va = seas < s, seas == s
    prior = np.bincount(y[tr], minlength=3) / tr.sum()
    filas.append(dict(fold=s, modelo='prior', bloque='-', ll=log_loss(y[va], np.tile(prior, (va.sum(), 1))), acc=accuracy_score(y[va], np.full(va.sum(), prior.argmax()))))
    filas.append(dict(fold=s, modelo='mercado', bloque='-', ll=log_loss(y[va], mk[va]), acc=accuracy_score(y[va], mk[va].argmax(1))))
ref = pd.DataFrame(filas)
print(ref.groupby('modelo').agg(ll_media=('ll', 'mean'), ll_std=('ll', 'std'), acc_media=('acc', 'mean')).round(4).to_string())
print('\nlog loss por fold (mercado):', ref[ref.modelo == 'mercado'].set_index('fold')['ll'].round(4).to_dict())

         ll_media  ll_std  acc_media
modelo                              
mercado    0.9665  0.0117     0.5383
prior      1.0726  0.0084     0.4369

log loss por fold (mercado): {1819: 0.9594, 1920: 0.9732, 2021: 0.9855, 2122: 0.9703, 2223: 0.9613, 2324: 0.9486, 2425: 0.9584, 2526: 0.9752}


**Hallazgo.**

| Referencia | Log loss (media) | Desvío entre folds | Accuracy |
|---|---|---|---|
| Prior | 1.0726 | 0.0084 | 0.437 |
| Mercado crudo | **0.9665** | 0.0117 | 0.538 |

El mercado mejora al prior en ≈ 0.106 de log loss: es la vara contra la que se mide cualquier modelo.

**Decisión.** Reportar siempre prior y mercado junto a los modelos.

## Paso 2 — Aporte de cada bloque de features

**Qué vamos a hacer.** Añadir las features por bloques y medir el log loss fuera de muestra:

| Bloque | Contenido |
|---|---|
| **M** | Mercado (`MKT_logit`, `MKT_pD`, `MKT_margin`) |
| **M+S** | + plantilla (`DIF_sq_top11`, `HomeTeam_sq_top11_mean`) |
| **M+S+G** | + goles/puntos/portería a cero de los últimos 5 partidos (= F1) |
| **M+S+G+E** | + tiros/córners/faltas/tarjetas (= F2, solo árboles) |

**Modelos.** Regresión logística (estandarización dentro del pipeline; solo bloques sin NaN) e HistGradientBoosting (tolera NaN). Hiperparámetros **fijos y conservadores**: es una medición comparativa, no el modelo final.

**Cómo se lee.** Para cada bloque se reporta la media y el desvío del log loss entre folds, y en cuántos folds mejora respecto al bloque anterior y respecto al mercado.

In [3]:
MKT = [f for f in ['MKT_logit', 'MKT_pD', 'MKT_margin'] if f in F2]
SQ = [f for f in ['DIF_sq_top11', 'HomeTeam_sq_top11_mean'] if f in F2]
FORMA = [f for f in F1 if f.startswith(('H_', 'A_'))]
ESTAD = [f for f in F2 if f not in F1]
BLOQUES = {'M': MKT, 'M+S': MKT + SQ, 'M+S+G (F1)': MKT + SQ + FORMA, 'M+S+G+E (F2)': MKT + SQ + FORMA + ESTAD}
print({k: len(v) for k, v in BLOQUES.items()})

def lr():
    return make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
def hgb():
    return HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_depth=3,
                                          min_samples_leaf=100, l2_regularization=1.0, random_state=42)

filas = []
for s in FOLDS:
    tr, va = seas < s, seas == s
    for nombre, cols in BLOQUES.items():
        for modelo, fab in (('LR', lr), ('HGB', hgb)):
            if modelo == 'LR' and df[cols].isna().any().any():
                continue                                   # LR no tolera NaN: solo bloques de F1
            m = fab().fit(df.loc[tr, cols], y[tr])
            p = m.predict_proba(df.loc[va, cols])
            filas.append(dict(fold=s, modelo=modelo, bloque=nombre, ll=log_loss(y[va], p), acc=accuracy_score(y[va], p.argmax(1))))
res = pd.concat([ref, pd.DataFrame(filas)], ignore_index=True)
res.to_csv(DATA + 'ablation_results.csv', index=False)

print(res.groupby(['modelo', 'bloque'], sort=False).agg(ll_media=('ll', 'mean'), ll_std=('ll', 'std'), acc_media=('acc', 'mean')).round(4).to_string())

print('\nMejora de log loss respecto al bloque anterior (media y folds en que mejora):')
orden = list(BLOQUES)
for modelo in ('LR', 'HGB'):
    for a, b in zip(orden[:-1], orden[1:]):
        ra = res[(res.modelo == modelo) & (res.bloque == a)].set_index('fold')['ll']
        rb = res[(res.modelo == modelo) & (res.bloque == b)].set_index('fold')['ll']
        if len(ra) and len(rb):
            d = ra - rb
            print(f'  {modelo:<3} {a:<14} -> {b:<14}: delta media = {d.mean():+.4f} | mejora en {(d > 0).sum()}/{len(d)} folds')
vs = res[res.modelo == 'mercado'].set_index('fold')['ll']
print('\nDiferencia respecto al mercado crudo (positivo = mejor que el mercado):')
for modelo in ('LR', 'HGB'):
    for nombre in BLOQUES:
        r = res[(res.modelo == modelo) & (res.bloque == nombre)].set_index('fold')['ll']
        if len(r):
            d = vs - r
            print(f'  {modelo:<3} {nombre:<14}: delta media = {d.mean():+.4f} | mejora en {(d > 0).sum()}/{len(d)} folds')

{'M': 3, 'M+S': 5, 'M+S+G (F1)': 13, 'M+S+G+E (F2)': 33}


                      ll_media  ll_std  acc_media
modelo  bloque                                   
prior   -               1.0726  0.0084     0.4369
mercado -               0.9665  0.0117     0.5383
LR      M               0.9655  0.0127     0.5379
HGB     M               0.9677  0.0126     0.5372
LR      M+S             0.9657  0.0129     0.5383
HGB     M+S             0.9681  0.0130     0.5364
LR      M+S+G (F1)      0.9662  0.0128     0.5391
HGB     M+S+G (F1)      0.9682  0.0130     0.5361
        M+S+G+E (F2)    0.9691  0.0130     0.5372

Mejora de log loss respecto al bloque anterior (media y folds en que mejora):
  LR  M              -> M+S           : delta media = -0.0002 | mejora en 1/8 folds
  LR  M+S            -> M+S+G (F1)    : delta media = -0.0005 | mejora en 1/8 folds
  HGB M              -> M+S           : delta media = -0.0004 | mejora en 2/8 folds
  HGB M+S            -> M+S+G (F1)    : delta media = -0.0001 | mejora en 4/8 folds
  HGB M+S+G (F1)     -> M+S+G+E (F2

**Hallazgo.** Log loss medio en los 8 folds (datos tras corregir el mapeo de clubes):

| Modelo | Bloque | Log loss | Accuracy |
|---|---|---|---|
| **LR** | **M** | **0.9655** | 0.538 |
| LR | M+S | 0.9657 | 0.538 |
| LR | M+S+G (F1) | 0.9662 | 0.539 |
| HGB | M | 0.9677 | 0.537 |
| HGB | M+S | 0.9680 | 0.536 |
| HGB | M+S+G (F1) | 0.9681 | 0.536 |
| HGB | M+S+G+E (F2) | 0.9691 | 0.538 |

1. **La plantilla (`top11_mean`) no mejora:** Δ = −0.0002 con LR (mejora en 1/8 folds) y −0.0004 con HGB (3/8).
2. **La forma (goles/puntos) no ayuda:** Δ = −0.0005 con LR (1/8 folds) y ≈ 0 con HGB (4/8).
3. **Las estadísticas de partido tampoco:** Δ = −0.0010 con HGB (2/8 folds).
4. **LR supera a HGB** en todos los bloques por ≈ 0.002–0.003.
5. **LR sobre solo el mercado mejora al mercado crudo en 0.0009** (6/8 folds): es una recalibración, no información nueva.

**Matiz.** Las diferencias entre bloques (0.0000–0.0010) son menores que la variación entre folds (± 0.012); lo relevante es el signo, sistemáticamente no positivo. Hiperparámetros fijos, sin tuning.

**Nota de datos.** Estos números ya incluyen la corrección del mapeo de clubes (110 colisiones, 9.9% de los partidos). Antes de corregirla la conclusión era la misma, pero la plantilla estaba subestimada para equipos como Arsenal o Inter.

**Decisión.** Con features agregadas de equipo (plantilla total y forma reciente) **no hay información adicional al mercado** en este esquema. Antes de entrenar más modelos, probar información *de otra naturaleza*: habilidades individuales del once titular y un target de regresión (diferencia de goles).

## Paso 3 — Habilidades individuales del once titular (clasificación)

**Qué vamos a hacer.** Añadir las 33 features `SK_*` (diferencia local − visitante de las habilidades del once titular, construidas en `feature.ipynb`, Paso 10) a los bloques anteriores y repetir la validación temporal:

| Bloque | Contenido |
|---|---|
| **M** | Mercado |
| **M+SK** | Mercado + 33 habilidades |
| **M+S+G+SK** | Mercado + plantilla + forma (F1) + habilidades |

Se prueban regresión logística con dos regularizaciones (`C = 1` y `C = 0.05`, porque son 36–46 features) e HistGradientBoosting.

**Sensibilidad a 2024/25.** El snapshot FC25 es de fin de temporada (julio de 2025, sin fichajes de verano), por lo que esa temporada podría tener cierto *look-ahead*. Se compara:

1. **con 2024/25** (los 8 folds y todas las filas),
2. **con 2024/25 en entrenamiento, pero evaluando solo en los 7 folds restantes** (para comparar sobre los mismos partidos),
3. **sin 2024/25**: las filas de esa temporada se quitan del entrenamiento y no hay fold de validación para ella.

Si el aporte de las habilidades no depende de cuál de las tres se use, el *look-ahead* no explica el resultado.

In [4]:
sk = pd.read_csv(DATA + 'dataset_skills.csv')
assert (sk[['Date', 'HomeTeam', 'AwayTeam']].values == df[['Date', 'HomeTeam', 'AwayTeam']].values).all()
SK = fs['SK_habilidades']
Xall = pd.concat([df.reset_index(drop=True), sk[SK]], axis=1)
GD = sk['GD'].to_numpy(float)
print('habilidades:', len(SK), '| filas:', len(Xall), '| NaN en SK:', int(sk[SK].isna().sum().sum()))

def lr_c(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
MODELOS_CLAS = {'LR C=1': lambda: lr_c(1.0), 'LR C=0.05': lambda: lr_c(0.05), 'HGB': hgb}
BLOQ_SK = {'M': MKT, 'M+SK': MKT + SK, 'M+S+G+SK': MKT + SQ + FORMA + SK}

def cv_clas(excluir_2425):
    filas = []
    keep = (seas != 2425) if excluir_2425 else np.ones(len(df), bool)
    for s in FOLDS:
        if excluir_2425 and s == 2425:
            continue
        tr, va = (seas < s) & keep, seas == s
        for nombre, cols in BLOQ_SK.items():
            for modelo, fab in MODELOS_CLAS.items():
                m = fab().fit(Xall.loc[tr, cols], y[tr])
                p = m.predict_proba(Xall.loc[va, cols])
                filas.append(dict(fold=int(s), modelo=modelo, bloque=nombre, ll=log_loss(y[va], p), acc=accuracy_score(y[va], p.argmax(1))))
    return pd.DataFrame(filas)

con = cv_clas(False)
sin = cv_clas(True)
con7 = con[con.fold != 2425]
orden_b = list(BLOQ_SK)
t = pd.DataFrame({
    '1) con 2024/25, 8 folds': con.groupby(['modelo', 'bloque']).ll.mean(),
    '2) con 2024/25 en train, 7 folds': con7.groupby(['modelo', 'bloque']).ll.mean(),
    '3) sin 2024/25, 7 folds': sin.groupby(['modelo', 'bloque']).ll.mean(),
}).round(4)
t = t.reindex(pd.MultiIndex.from_product([list(MODELOS_CLAS), orden_b], names=['modelo', 'bloque']))
print('Log loss medio (menor es mejor). Mercado crudo (7 folds):', round(ref[(ref.modelo == 'mercado') & (ref.fold != 2425)].ll.mean(), 4), '| (8 folds):', round(ref[ref.modelo == 'mercado'].ll.mean(), 4))
print(t.to_string())

print('\nAporte de las habilidades = log loss(M) - log loss(M+SK); positivo = mejora:')
for etiqueta, d in (('1) con 2024/25, 8 folds', con), ('2) con 2024/25 en train, 7 folds', con7), ('3) sin 2024/25, 7 folds', sin)):
    for modelo in MODELOS_CLAS:
        a = d[(d.modelo == modelo) & (d.bloque == 'M')].set_index('fold').ll
        b = d[(d.modelo == modelo) & (d.bloque == 'M+SK')].set_index('fold').ll
        dd = a - b
        print(f'  {etiqueta:<34} {modelo:<10}: delta = {dd.mean():+.4f} | mejora en {(dd > 0).sum()}/{len(dd)} folds')
con.assign(variante='con').pipe(lambda x: pd.concat([x, sin.assign(variante='sin')])).to_csv(DATA + 'skills_clasificacion.csv', index=False)

habilidades: 33 | filas: 36119 | NaN en SK: 0


Log loss medio (menor es mejor). Mercado crudo (7 folds): 0.9676 | (8 folds): 0.9665
                    1) con 2024/25, 8 folds  2) con 2024/25 en train, 7 folds  3) sin 2024/25, 7 folds
modelo    bloque                                                                                      
LR C=1    M                          0.9655                            0.9667                   0.9668
          M+SK                       0.9670                            0.9682                   0.9682
          M+S+G+SK                   0.9676                            0.9689                   0.9690
LR C=0.05 M                          0.9656                            0.9668                   0.9668
          M+SK                       0.9670                            0.9682                   0.9682
          M+S+G+SK                   0.9676                            0.9689                   0.9689
HGB       M                          0.9677                            0.9690              

**Hallazgo.** Log loss medio (menor es mejor; mercado crudo: 0.9665 en 8 folds y 0.9676 en 7):

| Modelo | Bloque | 1) con 2024/25, 8 folds | 2) 7 folds, 2024/25 en train | 3) sin 2024/25, 7 folds |
|---|---|---|---|---|
| LR C=1 | M | **0.9655** | 0.9667 | 0.9668 |
| LR C=1 | M+SK | 0.9670 | 0.9682 | 0.9682 |
| LR C=1 | M+S+G+SK | 0.9676 | 0.9689 | 0.9690 |
| HGB | M | 0.9677 | 0.9690 | 0.9689 |
| HGB | M+SK | 0.9683 | 0.9701 | 0.9701 |
| HGB | M+S+G+SK | 0.9685 | 0.9703 | 0.9703 |

- **Las habilidades empeoran el log loss:** Δ = −0.0014 con LR (mejoran en solo 1/8 folds) y −0.0007 con HGB (2/8). `C = 0.05` da lo mismo que `C = 1`.
- **El resultado no depende de 2024/25:** la caída es −0.0014/−0.0015/−0.0015 con LR y −0.0007/−0.0010/−0.0011 con HGB en las tres variantes. Por tanto el posible *look-ahead* del snapshot de fin de temporada **no explica** que no haya mejora.
- Añadir además plantilla y forma (M+S+G+SK) empeora un poco más.

**Interpretación.** Las 33 habilidades están muy correlacionadas entre sí y con el `overall` (media |r| = 0.73) y su información ya está en las cuotas: añaden varianza sin sesgo que corregir.

**Decisión.** Las habilidades **no** entran en los modelos de clasificación finales. Se conservan solo para los experimentos de regresión y apilado (Pasos 4–5), que miden su información de otra manera.

## Paso 4 — Regresión de la diferencia de goles

**Qué vamos a hacer.** Cambiar el *target*: en vez de H/D/A, predecir `GD = goles local − goles visitante` (regresión), como en Stübinger et al. La diferencia de goles contiene más información por partido que el signo (un 4–0 y un 1–0 son ambos "H").

**Modelos** (los del paper, salvo SVM, que con 30 mil filas es demasiado lento):

- **LIN:** regresión lineal con estandarización (*LIR* del paper).
- **RF:** Random Forest (100 árboles, `min_samples_leaf = 50`, `max_features = 0.33`).
- **HGB:** HistGradientBoosting (equivalente al *boosting* del paper).
- **ALL:** promedio de los tres (el *ensemble* del paper).

**Bloques de features.**

| Bloque | Contenido | Qué responde |
|---|---|---|
| **SK** | 33 habilidades | ¿Qué tan bien predicen la diferencia de goles *por sí solas*? |
| **S+G+SK** | plantilla + forma + habilidades (sin mercado) | ¿Y con todo lo no-mercado? |
| **M+SK** | mercado + habilidades | ¿Qué hace el modelo cuando ve las cuotas? |

**Cómo se convierte `ŷ` en probabilidades 1X2.** Con una regresión logística multinomial sobre `ŷ` entrenada en la **temporada inmediatamente anterior** a la de validación (con un modelo que solo vio temporadas previas a esa). Así no hay fuga y no se usa una predicción *in-sample* sobrecalibrada.

**Métricas.** RMSE y MAE de `GD` (referencias: predecir la media del entrenamiento y regresar solo sobre el mercado) y log loss de las probabilidades derivadas.

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

def lin():
    return make_pipeline(StandardScaler(), LinearRegression())
def rf():
    return RandomForestRegressor(n_estimators=100, min_samples_leaf=50, max_features=0.33, max_samples=0.5, n_jobs=-1, random_state=42)
def hgbr():
    return HistGradientBoostingRegressor(learning_rate=0.05, max_iter=200, max_depth=3, min_samples_leaf=100, l2_regularization=1.0, random_state=42)

REG = {'LIN': lin, 'RF': rf, 'HGB': hgbr}
BLOQ_REG = {'SK': SK, 'S+G+SK': SQ + FORMA + SK, 'M+SK': MKT + SK}
MKT_MIN = ['MKT_logit', 'MKT_pD']
temporadas = sorted(df['Season'].unique())
prev_de = {s: temporadas[temporadas.index(s) - 1] for s in temporadas[1:]}

def pred_reg(s, cols, fab, excluir_2425=False):
    """Devuelve (yhat de la temporada anterior, yhat de la temporada s). El modelo de la anterior solo ve temporadas previas a ella."""
    keep = (seas != 2425) if excluir_2425 else np.ones(len(df), bool)
    p = prev_de[s]
    tr_c, ca = (seas < p) & keep, (seas == p) & keep
    tr, va = (seas < s) & keep, seas == s
    m_c = fab().fit(Xall.loc[tr_c, cols], GD[tr_c])
    m = fab().fit(Xall.loc[tr, cols], GD[tr])
    return ca, m_c.predict(Xall.loc[ca, cols]), va, m.predict(Xall.loc[va, cols])

def ll_desde(feats_cal, y_cal, feats_va, y_va):
    c = LogisticRegression(C=10.0, max_iter=2000).fit(feats_cal, y_cal)
    return log_loss(y_va, c.predict_proba(feats_va), labels=[0, 1, 2])

mk_l = Xall['MKT_logit'].to_numpy()
filas, preds = [], {}
for s in FOLDS:
    va = seas == s
    tr = seas < s
    # referencias de regresion
    mu = GD[tr].mean()
    filas.append(dict(fold=int(s), modelo='media', bloque='-', rmse=np.sqrt(np.mean((GD[va] - mu) ** 2)), mae=np.mean(np.abs(GD[va] - mu)), ll_y=np.nan, ll_stack=np.nan))
    for nombre, cols in {'M (mercado)': MKT_MIN}.items():
        ca, yc, va_, yv = pred_reg(s, cols, lin)
        filas.append(dict(fold=int(s), modelo='LIN', bloque=nombre, rmse=np.sqrt(np.mean((GD[va_] - yv) ** 2)), mae=np.mean(np.abs(GD[va_] - yv)),
                          ll_y=ll_desde(yc.reshape(-1, 1), y[ca], yv.reshape(-1, 1), y[va_]), ll_stack=np.nan))
    for nombre, cols in BLOQ_REG.items():
        guardado = {}
        for modelo, fab in REG.items():
            ca, yc, va_, yv = pred_reg(s, cols, fab)
            guardado[modelo] = (yc, yv)
        guardado['ALL'] = (np.mean([guardado[m][0] for m in REG], axis=0), np.mean([guardado[m][1] for m in REG], axis=0))
        for modelo, (yc, yv) in guardado.items():
            ll_y = ll_desde(yc.reshape(-1, 1), y[ca], yv.reshape(-1, 1), y[va_])
            # apilado con el mercado: P(FTR | MKT_logit, yhat) vs P(FTR | MKT_logit)
            ll_st = ll_desde(np.c_[mk_l[ca], yc], y[ca], np.c_[mk_l[va_], yv], y[va_])
            filas.append(dict(fold=int(s), modelo=modelo, bloque=nombre, rmse=np.sqrt(np.mean((GD[va_] - yv) ** 2)), mae=np.mean(np.abs(GD[va_] - yv)), ll_y=ll_y, ll_stack=ll_st))
        preds[(s, nombre)] = guardado
    # referencia de apilado: solo mercado
    ll_m = ll_desde(mk_l[ca].reshape(-1, 1), y[ca], mk_l[va_].reshape(-1, 1), y[va_])
    filas.append(dict(fold=int(s), modelo='solo mercado', bloque='-', rmse=np.nan, mae=np.nan, ll_y=np.nan, ll_stack=ll_m))
reg = pd.DataFrame(filas)
reg.to_csv(DATA + 'skills_regresion.csv', index=False)

res_r = reg.groupby(['modelo', 'bloque'], sort=False).agg(rmse=('rmse', 'mean'), mae=('mae', 'mean'), ll_solo_yhat=('ll_y', 'mean')).round(4)
print(res_r.to_string())

                            rmse     mae  ll_solo_yhat
modelo       bloque                                   
media        -            1.8400  1.4294           NaN
LIN          M (mercado)  1.6106  1.2464        0.9679
             SK           1.6640  1.2893        0.9922
RF           SK           1.6637  1.2870        0.9922
HGB          SK           1.6642  1.2872        0.9924
ALL          SK           1.6623  1.2864        0.9916
LIN          S+G+SK       1.6510  1.2797        0.9872
RF           S+G+SK       1.6554  1.2811        0.9885
HGB          S+G+SK       1.6536  1.2800        0.9875
ALL          S+G+SK       1.6509  1.2782        0.9867
LIN          M+SK         1.6120  1.2477        0.9687
RF           M+SK         1.6151  1.2511        0.9704
HGB          M+SK         1.6125  1.2490        0.9692
ALL          M+SK         1.6117  1.2480        0.9687
solo mercado -               NaN     NaN           NaN


**Hallazgo.** Promedio en los 8 folds:

| Modelo | Bloque | RMSE de GD | MAE | Log loss con solo `ŷ` |
|---|---|---|---|---|
| (predecir la media) | — | 1.8400 | 1.4294 | — |
| LIN | solo mercado | **1.6106** | 1.2464 | **0.9679** |
| LIN | SK | 1.6640 | 1.2893 | 0.9922 |
| ALL | SK | 1.6623 | 1.2864 | 0.9916 |
| LIN | S+G+SK | 1.6510 | 1.2797 | 0.9872 |
| ALL | S+G+SK | 1.6509 | 1.2782 | 0.9867 |
| LIN | M+SK | 1.6120 | 1.2477 | 0.9687 |
| ALL | M+SK | 1.6117 | 1.2480 | 0.9687 |

1. **Las habilidades sí contienen información sobre los goles.** Solas bajan el RMSE de 1.840 a 1.662 (−9.7%) y producen probabilidades 1X2 con log loss 0.992, claramente mejores que el prior (1.0726).
2. **Pero el mercado predice mejor:** una regresión lineal sobre solo `MKT_logit` y `MKT_pD` da RMSE 1.611 (log loss 0.968), mejor que cualquier bloque sin cuotas (el mejor, S+G+SK con ALL, 1.651).
3. **Añadir habilidades al mercado no ayuda:** 1.6117–1.6125 vs 1.6106.
4. **Los tres modelos del paper rinden casi igual** (LIN ≈ RF ≈ HGB; el ensemble ALL mejora apenas 0.0001–0.0017 al mejor individual): la relación entre estas features y la diferencia de goles es prácticamente lineal.

**Decisión.** La regresión de la diferencia de goles es válida y más informativa por partido, pero con estas features **no supera** al mercado. Falta comprobar si `ŷ` aporta algo *sobre* el mercado (Paso 5).

## Paso 5 — ¿Las habilidades aportan información sobre el mercado?

**Qué vamos a hacer.** Es la pregunta central del proyecto. Para cada predicción de diferencia de goles `ŷ` (que **no** vio las cuotas, bloques `SK` y `S+G+SK`) comparar:

- `P(FTR | MKT_logit)`: el mercado solo, con el mismo calibrador;
- `P(FTR | MKT_logit, ŷ)`: el mercado **más** la predicción de goles.

Si `ŷ` contiene información que las cuotas no tienen, el segundo log loss debe ser menor. Ambas regresiones logísticas se entrenan en la misma temporada de calibración, así que la comparación es limpia.

**Cómo se lee.** `delta = ll(solo mercado) − ll(mercado + ŷ)`; positivo = la predicción de goles aporta.

In [6]:
base_m = reg[reg.modelo == 'solo mercado'].set_index('fold').ll_stack
print(f'Log loss del apilado con solo mercado (media {len(base_m)} folds): {base_m.mean():.4f}\n')
filas_s = []
for modelo in list(REG) + ['ALL']:
    for nombre in BLOQ_REG:
        r = reg[(reg.modelo == modelo) & (reg.bloque == nombre)].set_index('fold')
        d = base_m - r.ll_stack
        filas_s.append(dict(modelo=modelo, bloque=nombre, ll_mercado_mas_yhat=r.ll_stack.mean(), delta=d.mean(), folds_mejora=f'{(d > 0).sum()}/{len(d)}', rmse=r.rmse.mean()))
st = pd.DataFrame(filas_s).round(4)
print(st.to_string(index=False))

# correlacion de la prediccion de goles sin mercado con la implicita en el mercado (como medida de redundancia)
print('\nCorrelacion entre yhat (S+G+SK, ALL) y MKT_logit en la validacion:')
cors = []
for s in FOLDS:
    yv = preds[(s, 'S+G+SK')]['ALL'][1]
    cors.append(np.corrcoef(yv, mk_l[seas == s])[0, 1])
print('  media:', round(float(np.mean(cors)), 3), '| por fold:', [round(float(c), 2) for c in cors])

Log loss del apilado con solo mercado (media 8 folds): 0.9678

modelo bloque  ll_mercado_mas_yhat   delta folds_mejora   rmse
   LIN     SK               0.9686 -0.0008          1/8 1.6640
   LIN S+G+SK               0.9683 -0.0005          2/8 1.6510
   LIN   M+SK               0.9680 -0.0002          2/8 1.6120
    RF     SK               0.9683 -0.0006          3/8 1.6637
    RF S+G+SK               0.9682 -0.0005          3/8 1.6554
    RF   M+SK               0.9679 -0.0001          3/8 1.6151
   HGB     SK               0.9686 -0.0009          3/8 1.6642
   HGB S+G+SK               0.9682 -0.0004          2/8 1.6536
   HGB   M+SK               0.9678 -0.0000          3/8 1.6125
   ALL     SK               0.9685 -0.0008          3/8 1.6623
   ALL S+G+SK               0.9683 -0.0005          2/8 1.6509
   ALL   M+SK               0.9678  0.0000          3/8 1.6117

Correlacion entre yhat (S+G+SK, ALL) y MKT_logit en la validacion:
  media: 0.914 | por fold: [0.92, 0.91, 0.91, 0.91

**Hallazgo.** El log loss del apilado con solo el mercado es 0.9678 (el calibrador usa solo la temporada anterior; por eso es algo mayor que el 0.9655 de la regresión logística entrenada con todo el historial).

| Predicción de goles `ŷ` | Log loss de `MKT_logit` + `ŷ` | delta | Folds en que mejora |
|---|---|---|---|
| ALL, SK | 0.9685 | −0.0008 | 3/8 |
| ALL, S+G+SK | 0.9683 | −0.0005 | 2/8 |
| ALL, M+SK | 0.9678 | 0.0000 | 3/8 |
| LIN, SK | 0.9686 | −0.0008 | 1/8 |
| RF, SK | 0.9683 | −0.0006 | 3/8 |
| HGB, SK | 0.9686 | −0.0009 | 3/8 |

- **Ninguna predicción de goles mejora al mercado:** todos los *delta* están entre −0.0009 y 0.0000.
- **Redundancia:** la correlación entre `ŷ` (S+G+SK) y `MKT_logit` es **0.91** en todos los folds. La predicción de goles construida sin cuotas reproduce casi la misma ordenación de equipos que el mercado.

**Decisión / respuesta a la pregunta.** **Con estos datos, las habilidades individuales no aportan información adicional a las cuotas**, ni en clasificación ni en regresión, y el resultado es el mismo con y sin la temporada 2024/25. Es un resultado negativo bien sustentado.

## Paso 6 — Selección de hiperparámetros de SVM, XGBoost y MLP

**Qué vamos a hacer.** Elegir los hiperparámetros de los tres modelos que faltan con **el mismo procedimiento** que en `mercado.ipynb` (Paso 1):

1. Rejilla corta por modelo.
2. Validación temporal en la **ventana de ajuste** (2016/17 y 2017/18), disjunta de los folds de evaluación (2018/19–2025/26).
3. Criterio: menor log loss. Desempate: la configuración **más simple** si la diferencia es < 0.0005.
4. Valores congelados.

| Modelo | Features | Rejilla |
|---|---|---|
| **SVM** (kernel RBF, probabilidades por escalamiento de Platt interno) | F1, estandarizadas | `C ∈ {0.1, 1, 10}`, `gamma ∈ {0.01, 0.05, 0.2}` |
| **XGBoost** | F2 (tolera NaN) | `learning_rate ∈ {0.03, 0.1}`, `max_depth ∈ {2, 3, 4}`, `n_estimators ∈ {100, 300}`, `min_child_weight ∈ {5, 30}` |
| **MLP** (ReLU, *early stopping*) | F1, estandarizadas | capas `∈ {(16), (32, 16)}`, `alpha ∈ {0.001, 0.01, 0.1, 1}` |

**Notas de diseño.**

- El SVM con kernel RBF escala de forma cuadrática con las filas: se entrena con las **9,000 filas más recientes** (≈ 3 temporadas) en la selección y en la evaluación, para que sea tratable y comparable.
- El MLP es estocástico: cada configuración se promedia sobre **3 semillas** (en la selección y en la evaluación).
- Las referencias (LR sobre cuotas y HGB) usan los valores ya elegidos en `hiperparametros.json`.

In [7]:
from itertools import product
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore', category=FutureWarning)   # SVC(probability=True) esta deprecado desde scikit-learn 1.9 (se retira en 1.11)
HP = json.load(open(DATA + 'hiperparametros.json'))
X1 = Xall                                   # DataFrame con todas las features (F1 sin NaN, F2 con NaN)
X2 = Xall[F2].to_numpy(float)               # XGBoost tolera NaN
XF1 = Xall[F1].to_numpy(float)              # F1: sin NaN, para SVM y MLP
TUNE = [1617, 1718]
assert not set(TUNE) & set(int(s) for s in FOLDS)
CAP_SVM = 9000

def cap(tr_mask, n=CAP_SVM):
    ix = np.where(tr_mask)[0]
    return ix[-n:] if len(ix) > n else ix

def svm_fab(C, gamma):
    return make_pipeline(StandardScaler(), SVC(C=C, gamma=gamma, kernel='rbf', probability=True, random_state=42))
def mlp_fab(hidden, alpha, seed=0):
    return make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=hidden, alpha=alpha, activation='relu', max_iter=300, early_stopping=True,
                                                         validation_fraction=0.1, n_iter_no_change=15, random_state=seed))
def xgb_fab(learning_rate, max_depth, n_estimators, min_child_weight):
    return XGBClassifier(objective='multi:softprob', tree_method='hist', learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators,
                         min_child_weight=min_child_weight, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, random_state=42, n_jobs=8)

def prob_mlp(hidden, alpha, Xtr, ytr, Xva, seeds=(0, 1, 2)):
    return np.mean([mlp_fab(hidden, alpha, s).fit(Xtr, ytr).predict_proba(Xva) for s in seeds], axis=0)

def ll_tune(fn):
    out = []
    for s in TUNE:
        tr, va = seas < s, seas == s
        out.append(log_loss(y[va], fn(tr, va), labels=[0, 1, 2]))
    return float(np.mean(out))

def elegir(res, simple_keys):
    t = pd.DataFrame(res).sort_values('ll').reset_index(drop=True)
    cerca = t[t.ll <= t.ll.iloc[0] + 0.0005]
    return cerca.sort_values(simple_keys + ['ll'], ascending=[True] * len(simple_keys) + [True]).iloc[0].to_dict(), t

BEST6, TAB6 = {}, {}
# SVM
res = []
for C, g in product((0.1, 1, 10), (0.01, 0.05, 0.2)):
    res.append(dict(C=C, gamma=g, ll=ll_tune(lambda tr, va: svm_fab(C, g).fit(XF1[cap(tr)], y[cap(tr)]).predict_proba(XF1[va]))))
BEST6['SVM'], TAB6['SVM'] = elegir(res, ['gamma', 'C'])
# XGBoost
res = []
for lr_, d_, n_, w_ in product((0.03, 0.1), (2, 3, 4), (100, 300), (5, 30)):
    res.append(dict(learning_rate=lr_, max_depth=d_, n_estimators=n_, min_child_weight=w_,
                    ll=ll_tune(lambda tr, va: xgb_fab(lr_, d_, n_, w_).fit(X2[tr], y[tr]).predict_proba(X2[va]))))
BEST6['XGB'], TAB6['XGB'] = elegir(res, ['max_depth', 'n_estimators'])
# MLP
res = []
for h_, a_ in product(((16,), (32, 16)), (0.001, 0.01, 0.1, 1.0)):
    res.append(dict(hidden=str(h_), alpha=a_, ll=ll_tune(lambda tr, va: prob_mlp(h_, a_, XF1[tr], y[tr], XF1[va]))))
t6 = pd.DataFrame(res)
t6['n_capas'] = t6.hidden.map(lambda s: len(eval(s)))
BEST6['MLP'] = (lambda d: d[d.ll <= d.ll.min() + 0.0005].sort_values(['n_capas', 'alpha'], ascending=[True, False]).iloc[0].to_dict())(t6)
TAB6['MLP'] = t6.sort_values('ll').reset_index(drop=True)

for k, t in TAB6.items():
    print(f'\n== {k}: {len(t)} configuraciones | mejor ll = {t.ll.min():.4f} | peor = {t.ll.max():.4f} | rango = {t.ll.max() - t.ll.min():.4f}')
    print(t.head(3).round(4).to_string(index=False))
    print('   ELEGIDA:', {a: (round(b, 4) if isinstance(b, float) else b) for a, b in BEST6[k].items() if a != 'n_capas'})
HP6 = {'SVM': {k: float(v) for k, v in BEST6['SVM'].items()}, 'XGB': {k: (float(v)) for k, v in BEST6['XGB'].items()},
       'MLP': {'hidden': BEST6['MLP']['hidden'], 'alpha': float(BEST6['MLP']['alpha']), 'll': float(BEST6['MLP']['ll'])}}
json.dump(HP6, open(DATA + 'hiperparametros_modelos.json', 'w'), indent=1)


== SVM: 9 configuraciones | mejor ll = 0.9704 | peor = 1.0227 | rango = 0.0522
   C  gamma     ll
 0.1   0.01 0.9704
 1.0   0.01 0.9708
10.0   0.01 0.9715
   ELEGIDA: {'C': 0.1, 'gamma': 0.01, 'll': 0.9704}

== XGB: 24 configuraciones | mejor ll = 0.9487 | peor = 0.9917 | rango = 0.0429
 learning_rate  max_depth  n_estimators  min_child_weight     ll
          0.03          2           100                 5 0.9487
          0.03          2           100                30 0.9489
          0.03          2           300                 5 0.9491
   ELEGIDA: {'learning_rate': 0.03, 'max_depth': 2.0, 'n_estimators': 100.0, 'min_child_weight': 5.0, 'll': 0.9487}

== MLP: 8 configuraciones | mejor ll = 0.9457 | peor = 0.9482 | rango = 0.0025
  hidden  alpha     ll  n_capas
(32, 16)  0.001 0.9457        2
(32, 16)  1.000 0.9461        2
   (16,)  1.000 0.9463        1
   ELEGIDA: {'hidden': '(32, 16)', 'alpha': 1.0, 'll': 0.9461}


**Hallazgo.**

| Modelo | Configuraciones | Mejor | Peor | Rango | **Elegida** |
|---|---|---|---|---|---|
| SVM (RBF) | 9 | 0.9704 | 1.0227 | 0.0522 | `C = 0.1`, `gamma = 0.01` |
| XGBoost | 24 | 0.9487 | 0.9917 | 0.0429 | `learning_rate = 0.03`, `max_depth = 2`, `n_estimators = 100`, `min_child_weight = 5` |
| MLP | 8 | 0.9457 | 0.9482 | 0.0025 | capas `(32, 16)`, `alpha = 1.0` |

- **SVM y XGBoost son sensibles** (rangos de 0.04–0.05 de log loss) y los dos eligen la configuración **más suave** de su rejilla (el SVM con `C` y `gamma` mínimos; XGBoost con árboles de profundidad 2 y pocas iteraciones): con datos tan ruidosos, la complejidad sobreajusta.
- **El MLP es insensible** (rango 0.0025): cualquier configuración da lo mismo; se elige la regularización más fuerte (`alpha = 1`) por el criterio de simplicidad.
- Los tres se acercan a la LR en la ventana de ajuste (0.943), nunca la superan: la ventana pequeña no favorece a modelos con más parámetros.

**Decisión.** Congelar estos valores (guardados en `hiperparametros_modelos.json`) y evaluarlos en los 8 folds.

## Paso 7 — Evaluación de SVM, XGBoost, MLP y ensemble

**Qué vamos a hacer.** Con los hiperparámetros congelados, evaluar en los 8 folds temporales (2018/19 – 2025/26; entrenamiento con todas las temporadas anteriores) y comparar con:

- **Prior** y **mercado crudo** (referencias sin modelo).
- **LR sobre cuotas** (`C` elegido en `mercado.ipynb`) y **HGB sobre F1** (hiperparámetros elegidos allí).
- **ALL:** promedio de las probabilidades de LR, HGB, XGBoost (F2), SVM y MLP, como el *ensemble* del paper.

**Métricas.** Log loss (principal), RPS y accuracy, con media y desvío entre folds. Además, la **diferencia por partido** respecto a la LR sobre cuotas con intervalo por *bootstrap pareado* (1,000 remuestreos): es la forma correcta de saber si una diferencia de 0.001 es ruido.

**Por qué.** Es la comparación de modelos que pide el enunciado, pero con el criterio adecuado: no "cuál tiene el mejor número", sino "¿alguno es mejor que el mercado o que la LR sobre cuotas de forma distinguible del ruido?".

In [8]:
def rps(P, yy):
    O = np.eye(3)[yy]
    return float(np.mean(0.5 * (((np.cumsum(P, axis=1) - np.cumsum(O, axis=1))[:, :2]) ** 2).sum(axis=1)))

hp_h = HP['HGB_clas']
hgb_t = lambda: HistGradientBoostingClassifier(learning_rate=hp_h['learning_rate'], max_depth=int(hp_h['max_depth']), min_samples_leaf=int(hp_h['min_samples_leaf']),
                                                max_iter=int(hp_h['max_iter']), l2_regularization=1.0, random_state=42)
lr_t = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=HP['LR_clas']['C'], max_iter=5000))
hs, hx, hm = HP6['SVM'], HP6['XGB'], HP6['MLP']
hidden = eval(hm['hidden'])

nombres = ['prior', 'mercado', 'LR (cuotas)', 'HGB (F1)', 'XGBoost (F2)', 'SVM (F1)', 'MLP (F1)', 'ALL (promedio)']
OOF, filas = {n: [] for n in nombres}, []
idx_all, y_all, fold_all = [], [], []
for s in FOLDS:
    tr, va = seas < s, seas == s
    ix_cap = cap(tr)
    pri = np.bincount(y[tr], minlength=3) / tr.sum()
    P = {'prior': np.tile(pri, (va.sum(), 1)),
         'mercado': odds.loc[va, ['MKT_pH', 'MKT_pD', 'MKT_pA']].to_numpy(),
         'LR (cuotas)': lr_t().fit(X1.loc[tr, MKT], y[tr]).predict_proba(X1.loc[va, MKT]),
         'HGB (F1)': hgb_t().fit(X1.loc[tr, F1], y[tr]).predict_proba(X1.loc[va, F1]),
         'XGBoost (F2)': xgb_fab(hx['learning_rate'], int(hx['max_depth']), int(hx['n_estimators']), hx['min_child_weight']).fit(X2[tr], y[tr]).predict_proba(X2[va]),
         'SVM (F1)': svm_fab(hs['C'], hs['gamma']).fit(XF1[ix_cap], y[ix_cap]).predict_proba(XF1[va]),
         'MLP (F1)': prob_mlp(hidden, hm['alpha'], XF1[tr], y[tr], XF1[va])}
    P['ALL (promedio)'] = np.mean([P[k] for k in ('LR (cuotas)', 'HGB (F1)', 'XGBoost (F2)', 'SVM (F1)', 'MLP (F1)')], axis=0)
    for n in nombres:
        OOF[n].append(P[n])
        filas.append(dict(fold=int(s), modelo=n, ll=log_loss(y[va], P[n], labels=[0, 1, 2]), rps=rps(P[n], y[va]), acc=accuracy_score(y[va], P[n].argmax(1))))
    idx_all.append(np.where(va)[0]); y_all.append(y[va]); fold_all.append(np.full(va.sum(), int(s)))
OOF = {n: np.vstack(v) for n, v in OOF.items()}
yv, fv = np.concatenate(y_all), np.concatenate(fold_all)
rf = pd.DataFrame(filas)
tab = rf.groupby('modelo', sort=False).agg(ll=('ll', 'mean'), ll_std=('ll', 'std'), rps=('rps', 'mean'), acc=('acc', 'mean')).round(4)
print(tab.to_string())

# diferencia por partido frente a la LR sobre cuotas y frente al mercado (bootstrap pareado)
pl = lambda n: -np.log(np.clip(OOF[n][np.arange(len(yv)), yv], 1e-9, 1))
rng = np.random.default_rng(5)
def dif(a, b):
    d = pl(a) - pl(b)
    bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(1000)]
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5), float((d.mean() < 0))
print('\nDiferencia de log loss por partido (negativo = el primero es MEJOR); IC pareado 95%:')
filas2 = []
for n in nombres[3:]:
    for ref in ('LR (cuotas)', 'mercado'):
        m_, lo, hi, _ = dif(n, ref)
        fo = rf[rf.modelo == n].set_index('fold').ll - rf[rf.modelo == ref].set_index('fold').ll
        filas2.append(dict(modelo=n, vs=ref, delta=m_, ic=f'[{lo:+.4f}, {hi:+.4f}]', significativo=bool(hi < 0 or lo > 0), folds_mejor=f'{int((fo < 0).sum())}/{len(fo)}'))
print(pd.DataFrame(filas2).round(4).to_string(index=False))
pd.concat([pd.DataFrame({'idx': np.concatenate(idx_all), 'fold': fv, 'y': yv})] + [pd.DataFrame(OOF[n], columns=[f'{n}|{l}' for l in 'HDA']) for n in nombres], axis=1).to_csv(DATA + 'modelos_oof.csv', index=False)

                    ll  ll_std     rps     acc
modelo                                        
prior           1.0726  0.0084  0.2306  0.4369
mercado         0.9665  0.0117  0.1948  0.5383
LR (cuotas)     0.9656  0.0127  0.1946  0.5378
HGB (F1)        0.9676  0.0129  0.1950  0.5375
XGBoost (F2)    0.9694  0.0116  0.1955  0.5371
SVM (F1)        0.9927  0.0110  0.2008  0.5318
MLP (F1)        0.9675  0.0130  0.1951  0.5351
ALL (promedio)  0.9684  0.0121  0.1952  0.5374

Diferencia de log loss por partido (negativo = el primero es MEJOR); IC pareado 95%:


        modelo          vs  delta                 ic  significativo folds_mejor
      HGB (F1) LR (cuotas) 0.0021 [+0.0013, +0.0028]           True         0/8
      HGB (F1)     mercado 0.0011 [+0.0002, +0.0020]           True         2/8
  XGBoost (F2) LR (cuotas) 0.0039 [+0.0029, +0.0049]           True         0/8
  XGBoost (F2)     mercado 0.0030 [+0.0022, +0.0038]           True         0/8
      SVM (F1) LR (cuotas) 0.0271 [+0.0245, +0.0296]           True         0/8
      SVM (F1)     mercado 0.0262 [+0.0239, +0.0287]           True         0/8
      MLP (F1) LR (cuotas) 0.0019 [+0.0012, +0.0027]           True         1/8
      MLP (F1)     mercado 0.0010 [+0.0000, +0.0019]           True         4/8
ALL (promedio) LR (cuotas) 0.0028 [+0.0020, +0.0037]           True         0/8
ALL (promedio)     mercado 0.0019 [+0.0012, +0.0026]           True         0/8


**Hallazgo.** Media de los 8 folds temporales:

| Modelo | Log loss | Desvío | RPS | Accuracy |
|---|---|---|---|---|
| Prior | 1.0726 | 0.0084 | 0.2306 | 0.437 |
| Mercado crudo | 0.9665 | 0.0117 | 0.1948 | 0.538 |
| **LR (cuotas)** | **0.9656** | 0.0127 | **0.1946** | 0.538 |
| HGB (F1) | 0.9676 | 0.0129 | 0.1950 | 0.538 |
| MLP (F1) | 0.9675 | 0.0130 | 0.1951 | 0.535 |
| ALL (promedio) | 0.9684 | 0.0121 | 0.1952 | 0.537 |
| XGBoost (F2) | 0.9694 | 0.0116 | 0.1955 | 0.537 |
| SVM (F1) | 0.9927 | 0.0110 | 0.2008 | 0.532 |

*Diferencia de log loss por partido (positivo = el modelo es **peor**), con IC pareado al 95%:*

| Modelo | vs LR (cuotas) | vs mercado crudo | Folds en que supera a la LR |
|---|---|---|---|
| HGB | +0.0021 [+0.0013, +0.0028] | +0.0011 [+0.0002, +0.0020] | 0/8 |
| MLP | +0.0019 [+0.0012, +0.0027] | +0.0010 [+0.0000, +0.0019] | 1/8 |
| ALL | +0.0028 [+0.0020, +0.0037] | +0.0019 [+0.0012, +0.0026] | 0/8 |
| XGBoost | +0.0039 [+0.0029, +0.0049] | +0.0030 [+0.0022, +0.0038] | 0/8 |
| SVM | +0.0271 [+0.0245, +0.0296] | +0.0262 [+0.0239, +0.0287] | 0/8 |

1. **Ningún modelo supera a la LR sobre cuotas ni, de forma significativa, al mercado crudo.** Todos los intervalos están por encima de 0: son **peores**, no empatados.
2. **A mayor flexibilidad, peor:** LR (3 variables) < HGB ≈ MLP < ALL < XGBoost < SVM. La relación entre las cuotas y el resultado es casi lineal y el resto de features es ruido para estos modelos.
3. **El SVM es el peor por mucho** (+0.027), incluso con `C` y `gamma` elegidos: con 9,000 filas y el escalamiento de Platt interno, el SVM RBF produce probabilidades poco finas; se mantiene como evidencia de que no es el modelo adecuado para este problema.
4. **El ensemble no ayuda:** promediar cinco modelos con la LR no mejora a la LR; lo que añade es el ruido de los demás.

**Decisión.** El modelo recomendado es la **regresión logística sobre las cuotas**. Los modelos más complejos quedan como comparación del enunciado, con su resultado negativo documentado.

## Paso 8 — Calibración de los modelos finales

**Qué vamos a hacer.** Un modelo puede tener buen log loss y estar mal calibrado (o al revés). Se mide:

1. **ECE** (error de calibración esperado, 10 tramos) para cada resultado, y su promedio.
2. **Curva de fiabilidad** del resultado local para el mercado, la LR sobre cuotas y el ensemble.

**Por qué.** La calibración es lo que permite comparar con las cuotas para detectar valor; un modelo sobreconfiado pierde dinero aunque "acierte" bastante (lo que vimos con el modelo sin cuotas en `mercado.ipynb`).

In [9]:
import matplotlib.pyplot as plt

def ece(p, o, nb=10):
    b_ = np.minimum((p * nb).astype(int), nb - 1)
    return float(sum((b_ == k).mean() * abs(o[b_ == k].mean() - p[b_ == k].mean()) for k in np.unique(b_)))

filas = []
for n in nombres[1:]:
    e = [ece(OOF[n][:, j], (yv == j)) for j in range(3)]
    filas.append(dict(modelo=n, ECE_H=e[0], ECE_D=e[1], ECE_A=e[2], ECE_medio=float(np.mean(e))))
print(pd.DataFrame(filas).round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 5))
edges = np.linspace(0, 1, 11)
for n, st in (('mercado', 'k--'), ('LR (cuotas)', 'b-'), ('ALL (promedio)', 'r-')):
    p = OOF[n][:, 0]; o = (yv == 0)
    b_ = np.minimum((p * 10).astype(int), 9)
    xs = [p[b_ == k].mean() for k in range(10) if (b_ == k).sum() > 50]
    ys = [o[b_ == k].mean() for k in range(10) if (b_ == k).sum() > 50]
    ax.plot(xs, ys, st, marker='o', label=n)
ax.plot([0, 1], [0, 1], color='gray', lw=0.8)
ax.set_xlabel('Probabilidad predicha de victoria local'); ax.set_ylabel('Frecuencia observada'); ax.set_title('Curva de fiabilidad (8 folds de evaluacion)'); ax.legend(); plt.show()

        modelo  ECE_H  ECE_D  ECE_A  ECE_medio
       mercado 0.0144 0.0102 0.0157     0.0134
   LR (cuotas) 0.0091 0.0061 0.0060     0.0071
      HGB (F1) 0.0096 0.0084 0.0062     0.0081
  XGBoost (F2) 0.0208 0.0110 0.0194     0.0171
      SVM (F1) 0.0408 0.0073 0.0165     0.0215
      MLP (F1) 0.0126 0.0089 0.0065     0.0093
ALL (promedio) 0.0215 0.0133 0.0091     0.0147


/tmp/ipykernel_181874/3200359814.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel('Probabilidad predicha de victoria local'); ax.set_ylabel('Frecuencia observada'); ax.set_title('Curva de fiabilidad (8 folds de evaluacion)'); ax.legend(); plt.show()


**Hallazgo.**

| Modelo | ECE (H) | ECE (D) | ECE (A) | ECE medio |
|---|---|---|---|---|
| Mercado crudo | 0.0144 | 0.0102 | 0.0157 | 0.0134 |
| **LR (cuotas)** | **0.0091** | **0.0061** | **0.0060** | **0.0071** |
| HGB (F1) | 0.0096 | 0.0084 | 0.0062 | 0.0081 |
| MLP (F1) | 0.0126 | 0.0089 | 0.0065 | 0.0093 |
| ALL (promedio) | 0.0215 | 0.0133 | 0.0091 | 0.0147 |
| XGBoost (F2) | 0.0208 | 0.0110 | 0.0194 | 0.0171 |
| SVM (F1) | 0.0408 | 0.0073 | 0.0165 | 0.0215 |

- **La LR sobre cuotas reduce a la mitad el error de calibración del mercado** (0.0134 → 0.0071): es la recalibración del sesgo favorito–longshot vista en `mercado.ipynb`.
- **XGBoost y SVM están peor calibrados que el mercado** (0.017 y 0.022), y arrastran al ensemble (0.0147).
- Esto explica en parte el orden de log loss del Paso 7.

**Decisión.** Reportar la LR sobre cuotas como modelo final: mejor log loss, mejor RPS y mejor calibración, con tres variables.

## Paso 9 — Selección de características: Mutual Information vs. PCA

**Qué vamos a hacer.** Responder la pregunta de investigación 3: *¿qué técnica de reducción de dimensionalidad preserva mejor la señal predictiva: **MI** (seleccionar variables) o **PCA** (proyectar), como en Tammouch et al.?*

**Diseño.**

| Elemento | Detalle |
|---|---|
| **Dos conjuntos de variables** | **Con cuotas**: F1 + habilidades (46 variables, todas sin NaN). **Sin cuotas**: plantilla + forma + habilidades (43 variables). El segundo es el que se parece al escenario del paper: sin una variable dominante, donde la reducción puede importar |
| **Métodos** | Todas las variables; MI top-`k` con `k ∈ {5, 10, 20}`; PCA con `k ∈ {5, 10, 20}` componentes; PCA con 95% y 99% de varianza |
| **Sin fuga** | MI y PCA (con su estandarización) se calculan **solo con el entrenamiento de cada fold** |
| **Modelos** | LR (`C` elegido), HGB y MLP (hiperparámetros elegidos en `mercado.ipynb` y en el Paso 6) |
| **Elección de `k`** | Con la **ventana de ajuste** (2016/17–2017/18): para cada técnica y modelo se elige la `k` con menor log loss; se reporta en los 8 folds de evaluación |
| **Referencia** | *Todas las variables* y, para el conjunto con cuotas, el mercado |

**Por qué.** Tammouch et al. reportan que MI > PCA con 156 variables y 4 mil partidos, y que PCA descarta variables de baja varianza con valor predictivo. Aquí hay 36 mil partidos y 43–46 variables: es una réplica en un régimen distinto.

In [10]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA

CON = F1 + SK
SIN = SQ + FORMA + SK
CONJ = {'Con cuotas': CON, 'Sin cuotas': SIN}
METODOS = ['Todas'] + [f'MI-{k}' for k in (5, 10, 20)] + [f'PCA-{k}' for k in (5, 10, 20)] + ['PCA-95%', 'PCA-99%']
FAM = {'MI': ['MI-5', 'MI-10', 'MI-20'], 'PCA (k comp.)': ['PCA-5', 'PCA-10', 'PCA-20'], 'PCA (varianza)': ['PCA-95%', 'PCA-99%']}
FOLDS_ALL = list(TUNE) + [int(s) for s in FOLDS]

def mod_lr(): return lr_t()
def mod_hgb(): return hgb_t()
def mod_mlp(): return mlp_fab(hidden, hm['alpha'], 0)
MODS = {'LR': mod_lr, 'HGB': mod_hgb, 'MLP': mod_mlp}

def transformar(metodo, Xtr, Xva, mi):
    if metodo == 'Todas':
        return Xtr, Xva
    if metodo.startswith('MI-'):
        k = int(metodo.split('-')[1]); ix = np.argsort(mi)[::-1][:k]
        return Xtr[:, ix], Xva[:, ix]
    sc = StandardScaler().fit(Xtr)
    n = int(metodo.split('-')[1]) if 'comp' not in metodo and '%' not in metodo else float(metodo.split('-')[1].strip('%')) / 100
    p = PCA(n_components=n, svd_solver='full', random_state=42).fit(sc.transform(Xtr))
    return p.transform(sc.transform(Xtr)), p.transform(sc.transform(Xva))

filas, info_pca, mi_top = [], [], {}
for nom_c, cols in CONJ.items():
    Xc = Xall[cols].to_numpy(float)
    for s in FOLDS_ALL:
        tr, va = seas < s, seas == s
        mi = mutual_info_classif(Xc[tr], y[tr], random_state=42, n_neighbors=3, n_jobs=-1)
        if s == FOLDS_ALL[-1]:
            mi_top[nom_c] = pd.Series(mi, index=cols).sort_values(ascending=False).head(8)
        for metodo in METODOS:
            Xt, Xv = transformar(metodo, Xc[tr], Xc[va], mi)
            if metodo.startswith('PCA-') and s == FOLDS_ALL[-1]:
                info_pca.append(dict(conjunto=nom_c, metodo=metodo, componentes=Xt.shape[1]))
            for nom_m, fab in MODS.items():
                p_ = fab().fit(Xt, y[tr]).predict_proba(Xv)
                filas.append(dict(conjunto=nom_c, metodo=metodo, modelo=nom_m, fold=s, ll=log_loss(y[va], p_, labels=[0, 1, 2])))
sel = pd.DataFrame(filas)
sel['ventana'] = sel.fold.isin(TUNE)
sel.to_csv(DATA + 'seleccion_caracteristicas.csv', index=False)
print('componentes para 95% / 99% de varianza (ultimo fold):'); print(pd.DataFrame(info_pca).query("metodo in ['PCA-95%', 'PCA-99%']").to_string(index=False))
for k, v in mi_top.items():
    print(f'\nTop MI (ultimo fold) - {k}:', v.round(3).to_dict())

# elegir k por tecnica y modelo con la ventana de ajuste; reportar en los folds de evaluacion
ev = sel[~sel.ventana]; vt = sel[sel.ventana]
resumen = []
for nom_c in CONJ:
    for nom_m in MODS:
        base = ev[(ev.conjunto == nom_c) & (ev.modelo == nom_m) & (ev.metodo == 'Todas')].ll
        resumen.append(dict(conjunto=nom_c, modelo=nom_m, tecnica='Todas', elegido='Todas', ll=base.mean(), ll_std=base.std()))
        for fam, lst in FAM.items():
            v = vt[(vt.conjunto == nom_c) & (vt.modelo == nom_m) & (vt.metodo.isin(lst))].groupby('metodo').ll.mean()
            ele = v.idxmin()
            e = ev[(ev.conjunto == nom_c) & (ev.modelo == nom_m) & (ev.metodo == ele)].ll
            resumen.append(dict(conjunto=nom_c, modelo=nom_m, tecnica=fam, elegido=ele, ll=e.mean(), ll_std=e.std()))
resumen = pd.DataFrame(resumen)
print('\nLog loss medio en los 8 folds de evaluacion (k elegida en la ventana de ajuste):')
print(resumen.pivot_table(index=['conjunto', 'modelo'], columns='tecnica', values='ll').round(4).to_string())
print('\nk elegida:'); print(resumen.pivot_table(index=['conjunto', 'modelo'], columns='tecnica', values='elegido', aggfunc='first').to_string())
print('\nMalla completa (media en folds de evaluacion):')
print(ev.groupby(['conjunto', 'modelo', 'metodo'], sort=False).ll.mean().unstack('metodo')[METODOS].round(4).to_string())

componentes para 95% / 99% de varianza (ultimo fold):
  conjunto  metodo  componentes
Con cuotas PCA-95%           27
Con cuotas PCA-99%           39
Sin cuotas PCA-95%           25
Sin cuotas PCA-99%           37

Top MI (ultimo fold) - Con cuotas: {'MKT_logit': 0.102, 'DIF_sq_top11': 0.09, 'SK_reactions': 0.077, 'SK_positioning': 0.074, 'SK_ball_control': 0.074, 'SK_vision': 0.07, 'SK_standing_tackle': 0.069, 'SK_short_passing': 0.068}

Top MI (ultimo fold) - Sin cuotas: {'DIF_sq_top11': 0.083, 'SK_reactions': 0.076, 'SK_ball_control': 0.075, 'SK_dribbling': 0.07, 'SK_positioning': 0.069, 'SK_interceptions': 0.069, 'SK_standing_tackle': 0.067, 'SK_short_passing': 0.066}

Log loss medio en los 8 folds de evaluacion (k elegida en la ventana de ajuste):
tecnica                MI  PCA (k comp.)  PCA (varianza)   Todas
conjunto   modelo                                               
Con cuotas HGB     0.9682         0.9823          0.9782  0.9679
           LR      0.9678         0.9815  

**Hallazgo.** Log loss medio en los 8 folds de evaluación (`k` elegida en la ventana de ajuste; prior 1.0726, mercado 0.9665):

| Conjunto | Modelo | Todas | MI (`k` elegida) | PCA (`k` comp.) | PCA (varianza) |
|---|---|---|---|---|---|
| **Con cuotas** (46 vars) | LR | **0.9676** | 0.9678 (MI-5) | 0.9815 (PCA-20) | 0.9681 (99%) |
| | HGB | **0.9679** | 0.9682 (MI-5) | 0.9823 (PCA-20) | 0.9782 (99%) |
| | MLP | 0.9697 | **0.9688** (MI-10) | 0.9832 (PCA-10) | 0.9800 (95%) |
| **Sin cuotas** (43 vars) | LR | **0.9877** | 0.9909 (MI-5) | 0.9871 (PCA-10) | 0.9878 (95%) |
| | HGB | 0.9876 | 0.9908 (MI-5) | 0.9885 (PCA-5) | **0.9880** (95%) |
| | MLP | 0.9873 | 0.9906 (MI-5) | 0.9876 (PCA-10) | 0.9895 (95%) |

El PCA con 95% / 99% de varianza necesita 25–27 / 37–39 componentes: la variabilidad está repartida entre muchas variables correlacionadas.

**Hallazgo clave.** El resultado **depende de si hay una variable dominante**:

- **Con cuotas, MI gana de forma clara** (Δ = −0.014 en 8/8 folds, ver abajo). Es lógico: MI conserva `MKT_logit`, la variable que concentra la señal; el PCA la mezcla con 45 variables ruidosas y necesita ≥ 20 componentes (o el 99% de varianza) para recuperarla. Aun así **ninguna técnica supera a usar todas las variables**: MI-5 empata (0.9678 vs 0.9676) y es equivalente a "usar solo el mercado".
- **Sin cuotas, PCA queda ligeramente mejor que MI** (Δ = +0.002 a +0.004; MI mejor en solo 1–3 de 8 folds) y **ninguno supera a todas las variables**. El MI univariado elige las 5–20 variables con más dependencia individual, que son habilidades muy correlacionadas entre sí (`reactions`, `ball_control`, `short_passing`, `vision`, `positioning`, `dribbling`, `standing_tackle`: |r| entre 0.76 y 0.94, media 0.86), y descarta la forma y otras variables **complementarias** con poca MI individual. El PCA, en cambio, resume la varianza compartida y conserva algo de lo demás.

In [11]:
# comparacion pareada MI vs PCA con la k elegida en la ventana (por fold)
print('MI vs PCA (k comp.): diferencia de log loss por fold, MI - PCA (negativo = MI mejor):')
filas = []
for nom_c in CONJ:
    for nom_m in MODS:
        r_mi = resumen[(resumen.conjunto == nom_c) & (resumen.modelo == nom_m) & (resumen.tecnica == 'MI')].elegido.iloc[0]
        r_pc = resumen[(resumen.conjunto == nom_c) & (resumen.modelo == nom_m) & (resumen.tecnica == 'PCA (k comp.)')].elegido.iloc[0]
        a = ev[(ev.conjunto == nom_c) & (ev.modelo == nom_m) & (ev.metodo == r_mi)].set_index('fold').ll
        b = ev[(ev.conjunto == nom_c) & (ev.modelo == nom_m) & (ev.metodo == r_pc)].set_index('fold').ll
        d = a - b
        filas.append(dict(conjunto=nom_c, modelo=nom_m, MI=r_mi, PCA=r_pc, delta_medio=d.mean(), folds_MI_mejor=f'{int((d < 0).sum())}/{len(d)}'))
print(pd.DataFrame(filas).round(4).to_string(index=False))
# referencia: que tan lejos queda cada tecnica del prior y del mercado
pri = rf[rf.modelo == 'prior'].ll.mean(); mer = rf[rf.modelo == 'mercado'].ll.mean()
print(f'\nreferencias (8 folds): prior {pri:.4f} | mercado {mer:.4f}')

MI vs PCA (k comp.): diferencia de log loss por fold, MI - PCA (negativo = MI mejor):
  conjunto modelo    MI    PCA  delta_medio folds_MI_mejor
Con cuotas     LR  MI-5 PCA-20      -0.0138            8/8
Con cuotas    HGB  MI-5 PCA-20      -0.0141            8/8
Con cuotas    MLP MI-10 PCA-10      -0.0144            8/8
Sin cuotas     LR  MI-5 PCA-10       0.0038            1/8
Sin cuotas    HGB  MI-5  PCA-5       0.0024            3/8
Sin cuotas    MLP  MI-5 PCA-10       0.0031            3/8

referencias (8 folds): prior 1.0726 | mercado 0.9665


**Hallazgo.** Comparación pareada de MI vs PCA (`k` comp.), diferencia de log loss por fold (negativo = MI mejor):

| Conjunto | Modelo | MI | PCA | Δ medio | Folds en que MI es mejor |
|---|---|---|---|---|---|
| Con cuotas | LR | MI-5 | PCA-20 | **−0.0138** | 8/8 |
| Con cuotas | HGB | MI-5 | PCA-20 | **−0.0141** | 8/8 |
| Con cuotas | MLP | MI-10 | PCA-10 | **−0.0144** | 8/8 |
| Sin cuotas | LR | MI-5 | PCA-10 | +0.0038 | 1/8 |
| Sin cuotas | HGB | MI-5 | PCA-5 | +0.0024 | 3/8 |
| Sin cuotas | MLP | MI-5 | PCA-10 | +0.0031 | 3/8 |

**Respuesta a la pregunta de investigación 3.**

1. **MI > PCA solo cuando hay una variable dominante** (las cuotas), que es el escenario de Tammouch et al. Ahí se **replica** su hallazgo y por la misma razón que ellos dan: PCA descarta o diluye variables con información.
2. **Sin variable dominante, el resultado se invierte** (PCA ≥ MI por 0.002–0.004): no se replica.
3. **En ambos casos, la mejor opción es no reducir** (o reducir muy poco): con 36 mil partidos y 43–46 variables **no hay maldición de la dimensionalidad** y los modelos regularizados ya manejan la redundancia. La diferencia entre técnicas (0.002–0.014) es mucho menor que la ganancia sobre el prior (0.085–0.105).

**Decisión.** Mantener todas las variables en los modelos finales (o solo las cuotas, que es lo que se recomienda). La selección por MI queda como ranking descriptivo.

## Paso 10 — Preguntas de investigación e hipótesis

**Qué vamos a hacer.** Cerrar explícitamente las preguntas y las hipótesis planteadas en el informe con los mismos folds temporales y los mismos modelos:

| # | Pregunta / hipótesis | Prueba |
|---|---|---|
| 1 | ¿El desempeño es estable entre temporadas o se degrada (*concept drift*)? | Log loss por fold, tendencia de `modelo − mercado` y entrenar con todo el pasado vs. solo las últimas 3 temporadas |
| 2 | ¿La plantilla mejora la predicción respecto a solo estadísticas históricas de partido? | LR y HGB con forma (G), plantilla (S) y ambas, sin y con cuotas; diferencia por partido con IC pareado |
| (ii) | ¿La plantilla aporta más **al inicio de la temporada**, cuando la forma es escasa? | Mejora por partido de añadir S, agrupada por semana de la temporada |
| (iii) | ¿Un modelo mal calibrado puede acertar más pero perder dinero? | Accuracy, ECE y ROI del backtest para los cuatro modelos de probabilidad |

(La pregunta 4 —*value bets* con un ensemble— y la hipótesis (i) ya están respondidas en `mercado.ipynb` y en los Pasos 2 y 7.)

In [12]:
# --- 1) estabilidad temporal
tab_f = rf.pivot(index='fold', columns='modelo', values='ll')[['prior', 'mercado', 'LR (cuotas)', 'HGB (F1)']]
tab_f['LR - mercado'] = tab_f['LR (cuotas)'] - tab_f['mercado']
tab_f['mercado - prior'] = tab_f['mercado'] - tab_f['prior']
print('Log loss por fold:'); print(tab_f.round(4).to_string())
from scipy.stats import spearmanr, linregress
orden_f = np.arange(len(tab_f))
for col in ('mercado', 'LR (cuotas)', 'LR - mercado', 'mercado - prior'):
    lr_ = linregress(orden_f, tab_f[col].to_numpy())
    rho, pr = spearmanr(orden_f, tab_f[col].to_numpy())
    print(f'  tendencia de {col:<16}: pendiente por temporada = {lr_.slope:+.5f} (p = {lr_.pvalue:.3f}) | Spearman = {rho:+.2f} (p = {pr:.3f})')

# entrenar con todo el pasado vs. solo las ultimas 3 temporadas (LR sobre cuotas)
filas_w = []
for s in FOLDS:
    va = seas == s
    tr_all = seas < s
    tr_w = (seas < s) & (seas >= temporadas[max(temporadas.index(s) - 3, 0)])
    pa = lr_t().fit(X1.loc[tr_all, MKT], y[tr_all]).predict_proba(X1.loc[va, MKT])
    pw = lr_t().fit(X1.loc[tr_w, MKT], y[tr_w]).predict_proba(X1.loc[va, MKT])
    filas_w.append(dict(fold=int(s), todo_el_pasado=log_loss(y[va], pa), ultimas_3=log_loss(y[va], pw)))
w = pd.DataFrame(filas_w); w['dif'] = w.ultimas_3 - w.todo_el_pasado
print('\nLR sobre cuotas: todo el pasado vs. ultimas 3 temporadas'); print(w.round(4).to_string(index=False))
print('  diferencia media (ultimas 3 - todo):', round(float(w.dif.mean()), 5), '| folds donde las ultimas 3 son mejores:', int((w.dif < 0).sum()), '/', len(w))

Log loss por fold:
modelo   prior  mercado  LR (cuotas)  HGB (F1)  LR - mercado  mercado - prior
fold                                                                         
1819    1.0645   0.9594       0.9588    0.9626       -0.0006          -0.1051
1920    1.0672   0.9732       0.9731    0.9741       -0.0000          -0.0941
2021    1.0880   0.9855       0.9870    0.9898        0.0014          -0.1025
2122    1.0754   0.9703       0.9692    0.9712       -0.0010          -0.1051
2223    1.0609   0.9613       0.9603    0.9624       -0.0010          -0.0995
2324    1.0747   0.9486       0.9456    0.9474       -0.0030          -0.1262
2425    1.0738   0.9584       0.9571    0.9581       -0.0012          -0.1154
2526    1.0761   0.9752       0.9732    0.9753       -0.0020          -0.1009
  tendencia de mercado         : pendiente por temporada = -0.00099 (p = 0.622) | Spearman = -0.17 (p = 0.693)
  tendencia de LR (cuotas)     : pendiente por temporada = -0.00133 (p = 0.538) | Spearman


LR sobre cuotas: todo el pasado vs. ultimas 3 temporadas
 fold  todo_el_pasado  ultimas_3     dif
 1819          0.9588     0.9595  0.0007
 1920          0.9731     0.9734  0.0003
 2021          0.9870     0.9863 -0.0007
 2122          0.9692     0.9694  0.0002
 2223          0.9603     0.9619  0.0015
 2324          0.9456     0.9465  0.0009
 2425          0.9571     0.9567 -0.0004
 2526          0.9732     0.9731 -0.0001
  diferencia media (ultimas 3 - todo): 0.0003 | folds donde las ultimas 3 son mejores: 3 / 8


**Hallazgo.**

*Estabilidad entre temporadas.* El log loss del mercado oscila entre 0.9486 (2023/24) y 0.9855 (2020/21), con desvío 0.0117, **sin tendencia** (pendiente −0.001 por temporada, *p* = 0.62; Spearman −0.17, *p* = 0.69). La diferencia **LR − mercado** es ≤ 0 en 7 de 8 temporadas y su magnitud **crece** en el tiempo (de −0.0006 en 2018/19 a −0.0020 en 2025/26, con un máximo de −0.0030 en 2023/24; Spearman −0.81, *p* = 0.015; pendiente *p* = 0.083): con solo 8 puntos se interpreta con cautela, pero el signo es el contrario a un deterioro.

*Entrenar con todo el pasado vs. las últimas 3 temporadas.* La ventana corta no mejora: diferencia media +0.0003 y es mejor solo en 3 de 8 folds.

**Respuesta a la pregunta 1.** **No hay *concept drift* detectable**: el desempeño es estable, la ventaja del modelo sobre el mercado no se degrada y no hay beneficio en olvidar temporadas antiguas. La mayor variación (2020/21, 0.9855) es la temporada con estadios vacíos, es decir, un cambio en el azar del propio fútbol y no del modelo.

In [13]:
# --- 2) plantilla vs. forma (con y sin cuotas): vectores de perdida por partido
def perdidas(cols, modelo):
    out = np.full(len(y), np.nan)
    for s in FOLDS:
        tr, va = seas < s, seas == s
        fab = lr_t if modelo == 'LR' else hgb_t
        p_ = fab().fit(Xall.loc[tr, cols], y[tr]).predict_proba(Xall.loc[va, cols])
        out[va] = -np.log(np.clip(p_[np.arange(va.sum()), y[va]], 1e-9, 1))
    return out
BL = {'G (forma)': FORMA, 'S (plantilla)': SQ, 'S+G': SQ + FORMA, 'M': MKT, 'M+G': MKT + FORMA, 'M+S': MKT + SQ, 'M+S+G': MKT + SQ + FORMA}
LV = {(b, m): perdidas(c, m) for b, c in BL.items() for m in ('LR', 'HGB')}
ev_m = ~np.isnan(LV[('M', 'LR')])
print('Log loss medio (8 folds) por bloque:')
print(pd.DataFrame({m: {b: np.nanmean(LV[(b, m)]) for b in BL} for m in ('LR', 'HGB')}).round(4).to_string())
rng = np.random.default_rng(3)
def dpar(a, b, m):
    d = (LV[(a, m)] - LV[(b, m)])[ev_m]
    bs = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(1000)]
    return d.mean(), np.percentile(bs, 2.5), np.percentile(bs, 97.5)
print('\nEfecto de anadir la plantilla (S) a la forma (G) y al mercado; diferencia por partido (negativo = S ayuda), IC pareado 95%:')
filas = []
for a, b, etiqueta in (('S+G', 'G (forma)', 'sin cuotas: G -> S+G'), ('M+S+G', 'M+G', 'con cuotas: M+G -> M+S+G'), ('M+S', 'M', 'con cuotas: M -> M+S'), ('G (forma)', 'S (plantilla)', 'sin cuotas: S -> G (forma sola vs plantilla sola)')):
    for m in ('LR', 'HGB'):
        d, lo, hi = dpar(a, b, m)
        filas.append(dict(comparacion=etiqueta, modelo=m, delta=d, ic=f'[{lo:+.4f}, {hi:+.4f}]', significativo=bool(hi < 0 or lo > 0)))
print(pd.DataFrame(filas).round(4).to_string(index=False))

Log loss medio (8 folds) por bloque:
                   LR     HGB
G (forma)      1.0275  1.0299
S (plantilla)  0.9908  0.9903
S+G            0.9864  0.9873
M              0.9656  0.9671
M+G            0.9660  0.9675
M+S            0.9658  0.9673
M+S+G          0.9663  0.9676

Efecto de anadir la plantilla (S) a la forma (G) y al mercado; diferencia por partido (negativo = S ayuda), IC pareado 95%:


                                      comparacion modelo   delta                 ic  significativo
                             sin cuotas: G -> S+G     LR -0.0410 [-0.0446, -0.0373]           True
                             sin cuotas: G -> S+G    HGB -0.0426 [-0.0464, -0.0388]           True
                         con cuotas: M+G -> M+S+G     LR  0.0003 [-0.0000, +0.0005]          False
                         con cuotas: M+G -> M+S+G    HGB  0.0002 [-0.0002, +0.0005]          False
                             con cuotas: M -> M+S     LR  0.0002 [-0.0001, +0.0005]          False
                             con cuotas: M -> M+S    HGB  0.0002 [-0.0002, +0.0007]          False
sin cuotas: S -> G (forma sola vs plantilla sola)     LR  0.0367 [+0.0320, +0.0409]           True
sin cuotas: S -> G (forma sola vs plantilla sola)    HGB  0.0396 [+0.0351, +0.0438]           True


**Hallazgo.** Log loss medio en los 8 folds:

| Bloque | LR | HGB |
|---|---|---|
| G (forma) | 1.0275 | 1.0299 |
| S (plantilla) | 0.9908 | 0.9903 |
| S+G | 0.9864 | 0.9873 |
| M (cuotas) | **0.9656** | 0.9671 |
| M+G | 0.9660 | 0.9675 |
| M+S | 0.9658 | 0.9673 |
| M+S+G | 0.9663 | 0.9676 |

*Diferencia por partido (negativo = ayuda), IC pareado 95%:*

| Comparación | LR | HGB |
|---|---|---|
| Sin cuotas: G → **S+G** | **−0.0410** [−0.0446, −0.0373] | **−0.0426** [−0.0464, −0.0388] |
| Sin cuotas: plantilla sola vs. forma sola (G − S) | +0.0367 [+0.032, +0.041] | +0.0396 [+0.035, +0.044] |
| Con cuotas: M+G → M+S+G | +0.0003 [−0.0000, +0.0005] | +0.0002 [−0.0002, +0.0005] |
| Con cuotas: M → M+S | +0.0002 [−0.0001, +0.0005] | +0.0002 [−0.0002, +0.0007] |

**Respuesta a la pregunta de investigación 2.** **Sí: la plantilla mejora la predicción respecto a solo estadísticas históricas de partido, y por un margen enorme**: sin cuotas, la plantilla sola obtiene 0.9908 contra 1.0275 de la forma sola (la forma de 5 partidos apenas supera al prior, 1.0726), y añadirla a la forma reduce el log loss en 0.041 [−0.045, −0.037]. **Pero esa mejora desaparece por completo en presencia de las cuotas** (Δ ≈ +0.0002, no significativo): las cuotas ya contienen toda la información de la plantilla.

In [14]:
# --- (ii) la plantilla aporta mas al inicio de la temporada?
fechas = pd.to_datetime(df['Date'], dayfirst=True, format='mixed')
inicio = fechas.groupby([df['Div'], df['Season']]).transform('min')
semana = ((fechas - inicio).dt.days // 7).to_numpy()
grupos = pd.cut(semana, [-1, 5, 11, 23, 100], labels=['semanas 1-6', 'semanas 7-12', 'semanas 13-24', 'semanas 25+'])
print('Partidos de evaluacion por grupo:', pd.Series(grupos[ev_m]).value_counts().sort_index().to_dict())
def por_grupo(a, b, m):
    d = LV[(a, m)] - LV[(b, m)]
    out = []
    for g in grupos.categories:
        mk_ = ev_m & (grupos == g)
        x = d[mk_]
        bs = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(500)]
        out.append(dict(grupo=g, n=int(mk_.sum()), delta=x.mean(), ic=f'[{np.percentile(bs, 2.5):+.4f}, {np.percentile(bs, 97.5):+.4f}]'))
    return pd.DataFrame(out)
for a, b, et in (('M+S', 'M', 'con cuotas: efecto de anadir plantilla (M+S - M), LR'), ('S+G', 'G (forma)', 'sin cuotas: efecto de anadir plantilla (S+G - G), LR')):
    print(f'\n{et}  [negativo = la plantilla ayuda]')
    print(por_grupo(a, b, 'LR').round(4).to_string(index=False))
# la forma es menos informativa al inicio?
print('\nPerdida del modelo con forma sola (G), por grupo de semanas (LR):')
print(pd.Series({g: np.nanmean(LV[('G (forma)', 'LR')][ev_m & (grupos == g)]) for g in grupos.categories}).round(4).to_string())
print('Perdida del prior-equivalente (mercado crudo), por grupo:')
pm_ = -np.log(np.clip(odds[['MKT_pH', 'MKT_pD', 'MKT_pA']].to_numpy()[np.arange(len(y)), y], 1e-9, 1))
print(pd.Series({g: float(pm_[ev_m & (grupos == g)].mean()) for g in grupos.categories}).round(4).to_string())

Partidos de evaluacion por grupo: {'semanas 1-6': 3608, 'semanas 7-12': 3396, 'semanas 13-24': 6725, 'semanas 25+': 10454}

con cuotas: efecto de anadir plantilla (M+S - M), LR  [negativo = la plantilla ayuda]
        grupo     n   delta                 ic
  semanas 1-6  3608 -0.0000 [-0.0008, +0.0006]
 semanas 7-12  3396 -0.0001 [-0.0007, +0.0006]
semanas 13-24  6725  0.0005 [-0.0000, +0.0011]
  semanas 25+ 10454  0.0002 [-0.0003, +0.0006]

sin cuotas: efecto de anadir plantilla (S+G - G), LR  [negativo = la plantilla ayuda]
        grupo     n   delta                 ic
  semanas 1-6  3608 -0.0557 [-0.0655, -0.0470]
 semanas 7-12  3396 -0.0404 [-0.0497, -0.0309]
semanas 13-24  6725 -0.0369 [-0.0437, -0.0299]
  semanas 25+ 10454 -0.0388 [-0.0441, -0.0336]

Perdida del modelo con forma sola (G), por grupo de semanas (LR):
semanas 1-6      1.0399
semanas 7-12     1.0182
semanas 13-24    1.0301
semanas 25+      1.0245
Perdida del prior-equivalente (mercado crudo), por grupo:
semanas 1-6 

**Hallazgo.** Partidos por grupo de semanas desde el inicio de cada liga-temporada: 1–6: 3,608; 7–12: 3,396; 13–24: 6,725; 25+: 10,454.

| Grupo | Efecto de añadir la plantilla **con cuotas** (M+S − M) | Efecto **sin cuotas** (S+G − G) |
|---|---|---|
| Semanas 1–6 | −0.0000 [−0.0008, +0.0006] | **−0.0557** [−0.0655, −0.0470] |
| Semanas 7–12 | −0.0001 [−0.0007, +0.0006] | −0.0404 [−0.0497, −0.0309] |
| Semanas 13–24 | +0.0005 [−0.0000, +0.0011] | −0.0369 [−0.0437, −0.0299] |
| Semanas 25+ | +0.0002 [−0.0003, +0.0006] | −0.0388 [−0.0441, −0.0336] |

El modelo con solo forma pierde más al inicio (1.0399 en las semanas 1–6 frente a 1.0182–1.0301 después).

**Respuesta a la hipótesis (ii).**

- **La parte "sobre todo al inicio de la temporada" se cumple, pero solo sin cuotas:** la plantilla aporta un 40% más en las semanas 1–6 (−0.056 frente a −0.037/−0.040), precisamente cuando la forma de los equipos aún no existe.
- **La parte "aportan señal complementaria" no se cumple:** con las cuotas, el efecto de la plantilla es ≈ 0 en **todos** los grupos, incluido el inicio de la temporada. El mercado ya incorpora esa información desde la primera jornada.

In [15]:
# --- (iii) calibracion vs. dinero: accuracy, ECE y ROI del backtest
oo = pd.read_csv(DATA + 'predicciones_oof.csv')
bt_r = pd.read_csv(DATA + 'backtest_resultados.csv')
yy_ = oo['y'].to_numpy()
filas = []
for k, nom in (('NM', 'Sin cuotas (goles -> prob.)'), ('lr', 'LR sobre cuotas'), ('hgb', 'HGB sobre F1'), ('stack', 'Apilado (mercado + y_NM)')):
    P_ = oo[[f'P_{k}_H', f'P_{k}_D', f'P_{k}_A']].to_numpy()
    e = float(np.mean([ece(P_[:, j], (yy_ == j)) for j in range(3)]))
    r_b = bt_r[(bt_r.casa == 'B365') & (bt_r.modelo == k) & (bt_r.tau == 0.0)].iloc[0]
    filas.append(dict(modelo=nom, acc_1x2=accuracy_score(yy_, P_.argmax(1)), log_loss=log_loss(yy_, P_), ECE=e, apuestas_test=int(r_b.n_test), ROI_B365_tau0=r_b.roi_test, EV_prometido=r_b.ev_prometido))
print(pd.DataFrame(filas).round(4).to_string(index=False))

                     modelo  acc_1x2  log_loss    ECE  apuestas_test  ROI_B365_tau0  EV_prometido
Sin cuotas (goles -> prob.)   0.5233    0.9870 0.0096          10448        -0.0952        0.1983
            LR sobre cuotas   0.5378    0.9656 0.0071           2071         0.0513        0.0208
               HGB sobre F1   0.5375    0.9676 0.0081           4896        -0.0601        0.0521
   Apilado (mercado + y_NM)   0.5368    0.9683 0.0102           7443        -0.0348        0.0556


**Hallazgo.**

| Modelo | Accuracy 1X2 | Log loss | ECE | Apuestas (test, τ = 0) | ROI en B365 | EV prometido |
|---|---|---|---|---|---|---|
| Sin cuotas (goles → prob.) | 52.3% | 0.9870 | 0.0096 | 10,448 | **−9.5%** | +19.8% |
| HGB sobre F1 | 53.8% | 0.9676 | 0.0081 | 4,896 | −6.0% | +5.2% |
| Apilado | 53.7% | 0.9683 | 0.0102 | 7,443 | −3.5% | +5.6% |
| LR sobre cuotas | 53.8% | 0.9656 | 0.0071 | 2,071 | +5.1% (IC [−1.8%, +11.8%]) | +2.1% |

**Respuesta a la hipótesis (iii).** El patrón observado **no es exactamente el que planteaba la hipótesis**:

- El modelo que pierde dinero (sin cuotas) **no es el peor calibrado**: su ECE (0.0096) es **mejor que el del mercado crudo** (0.0134) y comparable al de los demás. Tampoco "acierta más": acierta 1.5 pp **menos**.
- Lo que explica el −9.5% **no es la calibración promedio sino su calibración condicional en los partidos donde discrepa del mercado**: promete +19.8% de valor y obtiene −9.5% (la "maldición del ganador": las apuestas que elige son justo aquellas donde su error es mayor).
- Una diferencia de 1.5 pp de accuracy se traduce en una diferencia de **14.6 pp de ROI** (de −9.5% a +5.1%), por lo que la accuracy es una métrica casi inútil para valorar una estrategia de apuestas.

**Conclusión.** Una buena calibración global no garantiza rentabilidad: lo que importa es no tener errores sistemáticos donde se apuesta.

## Resumen y siguiente paso

**Preguntas de investigación (informe, sección 1).**

| # | Pregunta | Respuesta | Dónde |
|---|---|---|---|
| 1 | ¿Estable entre temporadas o *concept drift*? | **Estable**; la ventaja sobre el mercado no se degrada (incluso crece algo); entrenar con ventana corta no ayuda | Paso 10 |
| 2 | ¿La plantilla mejora la predicción frente a solo estadísticas históricas? | **Sí, sin cuotas** (Δ = −0.041, IC [−0.045, −0.037]); **no con cuotas** (Δ ≈ +0.0002) | Paso 10 |
| 3 | ¿MI o PCA preservan mejor la señal? | **MI con cuotas** (−0.014, 8/8 folds); **PCA ≥ MI sin cuotas** (+0.002 a +0.004); ninguna supera a usar todas las variables | Paso 9 |
| 4 | ¿Un ensemble de modelos calibrados detecta *value bets* rentables? | **No**: ROI −8.5% a −14.8% sin cuotas; el ensemble ALL empeora a la LR (+0.0028); nada tiene un IC por encima de 0 | `mercado.ipynb` Paso 7; Paso 7 |

**Hipótesis.**

| # | Hipótesis | Veredicto |
|---|---|---|
| (i) | Las cuotas dominan sobre las demás features | **Confirmada** (Pasos 2–5; `mercado.ipynb`) |
| (ii) | La plantilla aporta señal complementaria, sobre todo al inicio de la temporada | **Parcial**: la plantilla pesa más al inicio **sin cuotas**, pero **no es complementaria a las cuotas** en ningún momento |
| (iii) | Un modelo mal calibrado puede acertar más pero perder dinero | **No en esa forma**: el modelo perdedor está bien calibrado en promedio y acierta menos; el problema es su calibración condicional donde discrepa del mercado |

**Otros resultados.**

| Pregunta | Respuesta |
|---|---|
| ¿Los modelos superan al mercado crudo? | Solo la **LR sobre cuotas**, por 0.0009 (recalibración); SVM, XGBoost, MLP, HGB y el ensemble son significativamente peores |
| ¿Aportan las habilidades individuales (Stübinger et al.)? | **No:** empeoran el log loss y no mejoran al mercado al apilarlas |
| ¿Importa la temporada 2024/25 (snapshot de fin de temporada)? | No: el resultado es el mismo con y sin ella |
| ¿Qué modelo se recomienda? | **LR sobre `MKT_logit`, `MKT_pD`, `MKT_margin`**: mejor log loss (0.9656), RPS (0.1946) y calibración (ECE 0.0071) |

**Sobre los hiperparámetros.** Los de los Pasos 2–5 se fijaron de forma conservadora porque allí solo se compara el aporte de cada bloque. La **selección formal** (rejilla, ventana de ajuste disjunta de los folds de evaluación y desempate por simplicidad) está en `mercado.ipynb` (Paso 1: LR, HGB, RF) y en el Paso 6 de este notebook (SVM, XGBoost, MLP); los valores quedan en `hiperparametros.json` y `hiperparametros_modelos.json`.

**Siguiente.** Actualizar el informe (secciones 3–9) con estos resultados y corregir la cita del paper.